# 多模型负荷分解实验复现（UK-DALE House 1）

统一配置入口：修改第 1 节的 `CONFIG` 即可切换模型（Seq2Point / CNN3 / CNN5 / CNN7 / WaveNet / BGRU 等）与目标电器，
训练完成后自动输出训练曲线、预测可视化与误差统计。

> 基于姜洁老师 [fast-seq2point](https://github.com/jiejiang-jojo/fast-seq2point) 框架整理并改造为可配置的 Notebook 实验流程。


## 1. 实验参数配置

In [ ]:
# ============================================================
# 当前实验配置 - WaveNet 模型
# ============================================================
CONFIG = {
    'model': 'WaveNet',              # 模型名称
    'model_params': {
        'layers': 6,
        'kernel_size': 3,
        'residual_channels': 32,
        'dilation_channels': 32,
        'skip_channels': 32,         # WaveNet特有参数
    },
    'batch_size': 128,
    'width': 100,
    'validate_max_iteration': 2000,
    'max_iterations': 300000,        # 最大训练迭代次数
    'target_device': 'washingmachine',  # 目标电器
    'train_house_list': ['house11'],
    'validate_house_list': ['house11'],
    'inference_house': 'house11',
    'binary_threshold': None,         # None表示回归任务，数值表示二值化阈值
    'learning_rate': 1e-3,
    'lr_decay_step': 1000,
    'lr_decay_rate': 0.9,
    'min_lr': 5e-5,
    'checkpoint_freq': 1000,          # 模型保存频率
    'eval_freq': 1000,               # 验证频率
}

# ============================================================
# 其他模型配置（注释形式）
# ============================================================

# --- Seq2Point 模型 ---
# CONFIG_Seq2Point = {
#     'model': 'Seq2Point',
#     'model_params': {
#         'seq_len': 15,              # seq_len 必须 >= 10
#     },
#     'batch_size': 128,
#     'width': 100,
#     'validate_max_iteration': 2000,
#     'max_iterations': 300000,
#     'target_device': 'washingmachine',
#     'train_house_list': ['house2'],
#     'validate_house_list': ['house2'],
#     'inference_house': 'house2',
#     'binary_threshold': None,
# }

# --- CNN3 模型 (seq_len 必须为奇数) ---
# CONFIG_CNN3 = {
#     'model': 'CNN3',
#     'model_params': {
#         'seq_len': 41,              # (seq_len - 1) % 2 == 0
#     },
#     'batch_size': 128,
#     'width': 100,
#     'validate_max_iteration': 2000,
#     'max_iterations': 300000,
#     'target_device': 'washingmachine',
#     'train_house_list': ['house2'],
#     'validate_house_list': ['house2'],
#     'inference_house': 'house2',
#     'binary_threshold': None,
# }

# --- CNN5 模型 ---
# CONFIG_CNN5 = {
#     'model': 'CNN5',
#     'model_params': {
#         'seq_len': 15,              # (seq_len - 3) % 4 == 0
#     },
#     'batch_size': 128,
#     'width': 100,
#     'validate_max_iteration': 2000,
#     'max_iterations': 300000,
#     'target_device': 'washingmachine',
#     'train_house_list': ['house2'],
#     'validate_house_list': ['house2'],
#     'inference_house': 'house2',
#     'binary_threshold': None,
# }

# --- CNN7 模型 ---
# CONFIG_CNN7 = {
#     'model': 'CNN7',
#     'model_params': {
#         'seq_len': 253,             # (seq_len - 1) % 6 == 0
#     },
#     'batch_size': 128,
#     'width': 100,
#     'validate_max_iteration': 2000,
#     'max_iterations': 300000,
#     'target_device': 'washingmachine',
#     'train_house_list': ['house2'],
#     'validate_house_list': ['house2'],
#     'inference_house': 'house2',
#     'binary_threshold': None,
# }

# --- BGRU 模型 ---
# CONFIG_BGRU = {
#     'model': 'BGRU',
#     'model_params': {
#         'seq_len': 511,
#     },
#     'batch_size': 128,
#     'width': 100,
#     'validate_max_iteration': 2000,
#     'max_iterations': 300000,
#     'target_device': 'washingmachine',
#     'train_house_list': ['house2'],
#     'validate_house_list': ['house2'],
#     'inference_house': 'house2',
#     'binary_threshold': None,
# }

# --- LSTM 模型 ---
# CONFIG_LSTM = {
#     'model': 'LSTM',
#     'model_params': {
#         'seq_len': 511,
#     },
#     'batch_size': 128,
#     'width': 100,
#     'validate_max_iteration': 2000,
#     'max_iterations': 300000,
#     'target_device': 'washingmachine',
#     'train_house_list': ['house2'],
#     'validate_house_list': ['house2'],
#     'inference_house': 'house2',
#     'binary_threshold': None,
# }

# --- WaveNetBGRU 模型 ---
# CONFIG_WaveNetBGRU = {
#     'model': 'WaveNetBGRU',
#     'model_params': {
#         'layers': 6,
#         'kernel_size': 3,
#         'residual_channels': 32,
#         'dilation_channels': 32,
#         'skip_channels': 32,
#     },
#     'batch_size': 128,
#     'width': 100,
#     'validate_max_iteration': 2000,
#     'max_iterations': 300000,
#     'target_device': 'washingmachine',
#     'train_house_list': ['house2'],
#     'validate_house_list': ['house2'],
#     'inference_house': 'house2',
#     'binary_threshold': None,
# }

# --- 二值化分类任务配置示例 ---
# CONFIG_BINARY = {
#     'model': 'WaveNet',
#     'model_params': {
#         'layers': 6,
#         'kernel_size': 3,
#         'residual_channels': 32,
#         'dilation_channels': 32,
#         'skip_channels': 32,
#     },
#     'batch_size': 128,
#     'width': 100,
#     'validate_max_iteration': 2000,
#     'max_iterations': 300000,
#     'target_device': 'washingmachine',
#     'train_house_list': ['house2'],
#     'validate_house_list': ['house2'],
#     'inference_house': 'house2',
#     'binary_threshold': 20.0,       # 设置二值化阈值
# }

## 2. 导入依赖

In [ ]:
import os
import sys
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import h5py

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from tqdm.notebook import tqdm
from safetensors.torch import save_file as save_safetensors

# 导入项目模块
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from utilities import (calculate_scalar, scale, inverse_scale, binarize,
                       mean_absolute_error, signal_aggregate_error,
                       precision, recall, f_value, roc_auc, average_precision,
                       tp_fn_fp_tn, create_folder)
from models import MODELS, move_data_to_gpu

## 3. 路径和数据配置

In [ ]:
# 路径配置
WORKSPACE = os.path.dirname(os.path.abspath('__file__'))
HDF5_PATH = os.path.join(WORKSPACE, 'data.h5')

# 输出目录 - 优化命名格式
EXPERIMENT_ID = f"{CONFIG['model']}_{CONFIG['target_device']}_{int(time.time())}"
MODEL_SAVE_DIR = os.path.join(WORKSPACE, 'outputs', 'models', EXPERIMENT_ID)
LOG_DIR = os.path.join(WORKSPACE, 'outputs', 'logs', EXPERIMENT_ID)

create_folder(MODEL_SAVE_DIR)
create_folder(LOG_DIR)

print(f"实验ID: {EXPERIMENT_ID}")
print(f"工作目录: {WORKSPACE}")
print(f"数据文件: {HDF5_PATH}")
print(f"模型保存目录: {MODEL_SAVE_DIR}")
print(f"日志目录: {LOG_DIR}")

## 4. 数据生成器定义

In [ ]:
class DataGenerator:
    """数据生成器类，参考 data_generator.py"""
    
    def __init__(self, hdf5_path, target_device, train_house_list, validate_house_list,
                 batch_size, seq_len, width, random_seed=1234, binary_threshold=None):
        self.target_device = target_device
        self.batch_size = batch_size
        assert seq_len % 2 == 1, "seq_len must be odd"
        self.seq_len = seq_len
        self.width = width
        self.random_state = np.random.RandomState(random_seed)
        self.validate_random_state = np.random.RandomState(1)
        self.validate_house_list = validate_house_list
        self.binary_threshold = binary_threshold
        
        # 加载数据
        self.hf = h5py.File(hdf5_path, 'r')
        (self.train_x, self.train_y) = self.read_data(self.hf, target_device, train_house_list)
        (self.validate_x, self.validate_y) = self.read_data(self.hf, target_device, validate_house_list)
        
        # 计算归一化参数
        (self.mean_x, self.std_x, self.max_x) = calculate_scalar(self.train_x)
        (self.mean_y, self.std_y, self.max_y) = calculate_scalar(self.train_y)
        
        print(f"mean_x: {self.mean_x:.4f}, std_x: {self.std_x:.4f}")
        print(f"mean_y: {self.mean_y:.4f}, std_y: {self.std_y:.4f}")
        
        # 训练索引
        self.train_indexes = np.arange(0, len(self.train_x) - seq_len - width + 2, width)
        self.train_indexes = self.valid_data(self.train_x, self.train_y, self.train_indexes)
        
        # 验证索引
        self.validate_indexes = np.arange(0, len(self.validate_x) - seq_len - width + 2, width)
        self.validate_indexes = self.valid_data(self.validate_x, self.validate_y, self.validate_indexes)
        
        print(f"训练样本数: {len(self.train_indexes)}")
    
    def valid_data(self, inputs, outputs, indexes):
        """移除无效数据"""
        full_indexes = np.arange(0, len(inputs) - self.seq_len - self.width + 2, 1)
        length = len(full_indexes)
        
        for i in range(length):
            if inputs[i] < outputs[i]:
                full_indexes[max(i - self.seq_len // 2, 0):min(i + self.seq_len // 2, length)] = -1
        
        valid_indexes = np.array([j for j in indexes if (full_indexes[j] != -1)])
        return valid_indexes
    
    def read_data(self, hf, target_device, house_list):
        """加载数据"""
        if len(house_list) == 0:
            return [], []
        
        aggregates, targets = [], []
        for house in house_list:
            aggregate = hf[house]['aggregate'][:]
            target = hf[house][target_device][:]
            aggregates.append(aggregate)
            targets.append(target)
        
        return np.concatenate(aggregates, axis=0), np.concatenate(targets, axis=0)
    
    def generate(self):
        """生成训练数据"""
        batch_size = self.batch_size
        indexes = np.array(self.train_indexes)
        self.random_state.shuffle(indexes)
        
        pointer = 0
        while True:
            if pointer >= len(indexes):
                pointer = 0
                self.random_state.shuffle(indexes)
            
            batch_indexes = indexes[pointer: pointer + batch_size]
            pointer += batch_size
            
            batch_x_indexes_2d = batch_indexes[:, None] + np.arange(self.seq_len + self.width - 1)
            batch_y_indexes_2d = batch_indexes[:, None] + np.arange(self.seq_len // 2, self.seq_len // 2 + self.width)
            
            batch_x = self.train_x[batch_x_indexes_2d]
            batch_y = self.train_y[batch_y_indexes_2d]
            
            # 归一化
            batch_x = self.transform(batch_x)
            if self.binary_threshold is not None:
                batch_y = binarize(batch_y, self.binary_threshold)
            else:
                batch_y = self.transform(batch_y)
            
            yield batch_x, batch_y
    
    def generate_validate(self, data_type, max_iteration, shuffle=True):
        """生成验证数据"""
        if data_type == 'train':
            indexes = np.array(self.train_indexes)
            x, y = self.train_x, self.train_y
        else:
            indexes = np.array(self.validate_indexes)
            x, y = self.validate_x, self.validate_y
        
        if shuffle:
            self.validate_random_state.shuffle(indexes)
        
        pointer = 0
        iteration = 0
        
        while pointer < len(indexes) and iteration < max_iteration:
            batch_indexes = indexes[pointer: pointer + self.batch_size]
            pointer += self.batch_size
            iteration += 1
            
            batch_x_indexes_2d = batch_indexes[:, None] + np.arange(self.seq_len + self.width - 1)
            batch_y_indexes_2d = batch_indexes[:, None] + np.arange(self.seq_len // 2, self.seq_len // 2 + self.width)
            
            batch_x = x[batch_x_indexes_2d]
            batch_y = y[batch_y_indexes_2d]
            
            batch_x = self.transform(batch_x)
            if self.binary_threshold is not None:
                batch_y = binarize(batch_y, self.binary_threshold)
            else:
                batch_y = self.transform(batch_y)
            
            yield batch_x, batch_y
    
    def transform(self, x):
        return scale(x, self.mean_x, self.std_x)
    
    def inverse_transform(self, x):
        return inverse_scale(x, self.mean_x, self.std_x)


class TestDataGenerator(DataGenerator):
    """推理数据生成器"""
    
    def __init__(self, hdf5_path, target_device, train_house_list, seq_len, steps, binary_threshold=None):
        DataGenerator.__init__(self,
                              hdf5_path=hdf5_path,
                              target_device=target_device,
                              train_house_list=train_house_list,
                              validate_house_list=[],
                              batch_size=None,
                              seq_len=seq_len,
                              width=1,
                              random_seed=None,
                              binary_threshold=binary_threshold)
        self.steps = steps
    
    def generate_inference(self, house):
        """生成推理数据"""
        x, y = self.read_data(self.hf, self.target_device, house_list=[house])
        self.target = y
        self.source = x
        x = self.pad_seq(x)
        
        index = 0
        while index + self.seq_len <= len(x):
            batch_x = x[index: index + self.seq_len + self.steps - 1]
            batch_x = batch_x[np.newaxis, :]
            batch_x = self.transform(batch_x)
            index += self.steps
            yield batch_x
    
    def pad_seq(self, x):
        pad_num = self.seq_len // 2
        return np.concatenate((np.zeros(pad_num), x, np.zeros(pad_num)))
    
    def get_target(self):
        if self.binary_threshold is not None:
            return binarize(self.target, self.binary_threshold)
        return self.target
    
    def get_source(self):
        return self.source
    
    def inverse_transform(self, x):
        return inverse_scale(x, self.mean_x, self.std_x)

## 5. 损失函数和评估指标

In [ ]:
def loss_func(output, target):
    """MAE损失函数"""
    assert output.shape == target.shape
    return torch.mean(torch.abs(output - target))


def loss_func_binary(output, target):
    """二值交叉熵损失"""
    assert output.shape == target.shape
    return F.binary_cross_entropy(output, target)


def mean_abs_error(outputs, targets):
    """计算MAE"""
    return np.mean(np.abs(outputs - targets))


def signal_agg_error(outputs, targets):
    """计算SAE"""
    return np.abs(np.sum(outputs) - np.sum(targets)) / np.sum(targets)


def binary_metrics(outputs, targets, threshold=0.5):
    """计算二值分类指标"""
    outputs_bin = ((outputs - threshold) > 0).astype('float')
    targets_bin = ((targets - threshold) > 0).astype('float')
    
    tp = np.sum((outputs_bin + targets_bin) > 1.5)
    fn = np.sum(targets_bin - outputs_bin > 0.5)
    fp = np.sum(outputs_bin - targets_bin > 0.5)
    tn = np.sum(outputs_bin + targets_bin < 0.5)
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    
    return {
        'tp': int(tp), 'fn': int(fn), 'fp': int(fp), 'tn': int(tn),
        'precision': f'{prec:.4f}', 'recall': f'{rec:.4f}', 'f1': f'{f1:.4f}'
    }

## 6. 前向传播和评估函数

In [ ]:
def forward(model, generate_func, cuda, has_target=True):
    """前向传播"""
    model.eval()
    outputs, targets = [], []
    
    with torch.no_grad():
        for data in generate_func:
            if has_target:
                batch_x, batch_y = data
                targets.append(batch_y)
            else:
                batch_x = data
            
            batch_x = move_data_to_gpu(batch_x, cuda)
            batch_output = model(batch_x)
            outputs.append(batch_output.data.cpu().numpy())
    
    outputs = np.concatenate(outputs, axis=0)
    if has_target:
        targets = np.concatenate(targets, axis=0)
        return outputs, targets
    return outputs


def evaluate(model, generator, data_type, max_iteration, cuda, binary=False):
    """评估模型"""
    generate_func = generator.generate_validate(data_type=data_type, max_iteration=max_iteration)
    outputs, targets = forward(model, generate_func, cuda, has_target=True)
    
    if binary:
        return binary_metrics(outputs, targets)
    else:
        outputs = generator.inverse_transform(outputs)
        targets = generator.inverse_transform(targets)
        mae = mean_abs_error(outputs, targets)
        return {'MAE': mae}


def inference(model, generator, house, cuda, binary_threshold=None):
    """推理函数"""
    generate_func = generator.generate_inference(house=house)
    outputs = forward(model, generate_func, cuda, has_target=False)
    outputs = np.concatenate([output[0] for output in outputs])
    
    if binary_threshold is not None:
        return outputs, binary_metrics(outputs, generator.get_target(), binary_threshold)
    else:
        outputs = generator.inverse_transform(outputs)
        targets = generator.get_target()
        source = generator.get_source()
        
        # 计算有效数据掩码
        valid_data = np.ones_like(source)
        for i in range(len(source)):
            if source[i] == 0 or source[i] < targets[i]:
                valid_data[i] = 0
        
        mae = mean_abs_error(outputs * valid_data, targets * valid_data)
        sae = signal_agg_error(outputs * valid_data, targets * valid_data)
        
        return outputs, {'MAE': mae, 'SAE': sae}

## 7. 训练函数

In [ ]:
def train_model(config, workspace, cuda=True, resume_path=None):
    """
    训练模型
    
    参数:
        config: 实验配置字典
        workspace: 工作目录
        cuda: 是否使用CUDA
        resume_path: 恢复训练的模型路径
    """
    print("=" * 60)
    print(f"开始训练: {config['model']} 模型")
    print("=" * 60)
    
    # 加载模型
    model_class, model_params = MODELS[config['model']]
    model = model_class(**{k: config['model_params'].get(k, None) for k in model_params})
    
    # 如果有恢复路径，加载权重
    if resume_path:
        print(f"从 {resume_path} 恢复训练...")
        checkpoint = torch.load(resume_path)
        model.load_state_dict(checkpoint['state_dict'])
    
    print(f"序列长度: {model.seq_len}")
    
    if cuda and torch.cuda.is_available():
        model.cuda()
        print("使用 CUDA 加速")
    else:
        cuda = False
        print("使用 CPU")
    
    # 数据生成器
    hdf5_path = os.path.join(workspace, 'data.h5')
    generator = DataGenerator(
        hdf5_path=hdf5_path,
        target_device=config['target_device'],
        train_house_list=config['train_house_list'],
        validate_house_list=config['validate_house_list'],
        batch_size=config['batch_size'],
        seq_len=model.seq_len,
        width=config['width'],
        binary_threshold=config.get('binary_threshold', None)
    )
    
    # 优化器
    learning_rate = config['learning_rate']
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.999))
    
    # 训练状态
    iteration = 0
    best_val_mae = float('inf')
    best_iteration = 0
    train_bgn_time = time.time()
    
    # 训练历史记录
    history = {
        'train_mae': [],
        'val_mae': [],
        'lr': [],
        'iteration': []
    }
    
    # 主训练循环
    pbar = tqdm(total=config['max_iterations'], desc='Training', unit='iter')
    
    for batch_x, batch_y in generator.generate():
        if iteration >= config['max_iterations']:
            break
        
        # 评估
        if iteration % config['eval_freq'] == 0 and iteration > 0:
            train_fin_time = time.time()
            
            # 评估训练集
            train_result = evaluate(
                model=model,
                generator=generator,
                data_type='train',
                max_iteration=config['validate_max_iteration'],
                cuda=cuda,
                binary=config.get('binary_threshold') is not None
            )
            
            # 评估验证集
            val_result = evaluate(
                model=model,
                generator=generator,
                data_type='validate',
                max_iteration=config['validate_max_iteration'],
                cuda=cuda,
                binary=config.get('binary_threshold') is not None
            )
            
            train_time = train_fin_time - train_bgn_time
            val_time = time.time() - train_fin_time
            
            print(f"\n迭代 {iteration}:")
            print(f"  训练: {train_result}")
            print(f"  验证: {val_result}")
            print(f"  训练时间: {train_time:.2f}s, 验证时间: {val_time:.2f}s")
            print(f"  学习率: {learning_rate:.6f}")
            
            # 记录历史
            history['train_mae'].append(train_result.get('MAE', train_result.get('f1', 0)))
            history['val_mae'].append(val_result.get('MAE', val_result.get('f1', 0)))
            history['lr'].append(learning_rate)
            history['iteration'].append(iteration)
            
            # 检查是否为最优模型
            val_mae = val_result.get('MAE', float('inf'))
            if val_mae < best_val_mae:
                best_val_mae = val_mae
                best_iteration = iteration
                
                # 保存最优模型 (safetensors格式)
                best_model_path = os.path.join(
                    MODEL_SAVE_DIR,
                    f"best_model_{config['model']}_{config['target_device']}.safetensors"
                )
                save_safetensors(model.state_dict(), best_model_path)
                print(f"  [最优模型] 已保存到: {best_model_path}")
            
            train_bgn_time = time.time()
        
        # 学习率衰减
        if iteration % config['lr_decay_step'] == 0 and iteration > 0 and learning_rate > config['min_lr']:
            learning_rate *= config['lr_decay_rate']
            for param_group in optimizer.param_groups:
                param_group['lr'] = learning_rate
        
        # 前向传播
        model.train()
        batch_x = move_data_to_gpu(batch_x, cuda)
        batch_y = move_data_to_gpu(batch_y, cuda)
        
        output = model(batch_x)
        
        # 计算损失
        if config.get('binary_threshold') is not None:
            loss = loss_func_binary(output, batch_y)
        else:
            loss = loss_func(output, batch_y)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 更新进度条
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'lr': f'{learning_rate:.2e}',
            'best': f'{best_val_mae:.4f}'
        })
        pbar.update(1)
        
        iteration += 1
    
    pbar.close()
    
    print("\n" + "=" * 60)
    print(f"训练完成!")
    print(f"最优验证MAE: {best_val_mae:.4f} (迭代 {best_iteration})")
    print("=" * 60)
    
    # 保存训练历史
    history_path = os.path.join(LOG_DIR, 'training_history.json')
    with open(history_path, 'w') as f:
        json.dump(history, f)
    print(f"训练历史已保存到: {history_path}")
    
    return model, history, best_val_mae, best_iteration

## 8. 执行训练

In [ ]:
# 检查CUDA是否可用
cuda_available = torch.cuda.is_available()
print(f"CUDA可用: {cuda_available}")
if cuda_available:
    print(f"GPU设备: {torch.cuda.get_device_name(0)}")

# 执行训练
trained_model, training_history, best_mae, best_iter = train_model(
    config=CONFIG,
    workspace=WORKSPACE,
    cuda=cuda_available,
    resume_path=None  # 设置路径可恢复训练
)

## 9. 训练过程可视化

In [ ]:
# 绘制训练曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE曲线
ax1 = axes[0]
ax1.plot(training_history['iteration'], training_history['train_mae'], 
         'b-', label='Train MAE', linewidth=2)
ax1.plot(training_history['iteration'], training_history['val_mae'], 
         'r-', label='Validation MAE', linewidth=2)
ax1.set_xlabel('Iteration', fontsize=12)
ax1.set_ylabel('MAE', fontsize=12)
ax1.set_title('Training and Validation MAE', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# 学习率曲线
ax2 = axes[1]
ax2.plot(training_history['iteration'], training_history['lr'], 
         'g-', linewidth=2)
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Learning Rate', fontsize=12)
ax2.set_title('Learning Rate Schedule', fontsize=14)
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"训练曲线已保存到: {os.path.join(LOG_DIR, 'training_curves.png')}")

## 10. 加载最优模型并进行推理可视化

In [ ]:
# 加载最优模型
best_model_path = os.path.join(
    MODEL_SAVE_DIR,
    f"best_model_{CONFIG['model']}_{CONFIG['target_device']}.safetensors"
)

print(f"加载最优模型: {best_model_path}")

# 重新实例化模型
model_class, model_params = MODELS[CONFIG['model']]
model = model_class(**{k: CONFIG['model_params'].get(k, None) for k in model_params})

# 加载safetensors格式的权重
from safetensors.torch import load_file
state_dict = load_file(best_model_path)
model.load_state_dict(state_dict)

if cuda_available:
    model.cuda()

print("模型加载完成!")

## 11. 预测效果可视化（反缩放后的原始数据）

In [ ]:
# 创建推理数据生成器
inference_generator = TestDataGenerator(
    hdf5_path=HDF5_PATH,
    target_device=CONFIG['target_device'],
    train_house_list=CONFIG['train_house_list'],
    seq_len=model.seq_len,
    steps=CONFIG['width'] * CONFIG['batch_size'],
    binary_threshold=CONFIG.get('binary_threshold')
)

# 执行推理
predictions, metrics = inference(
    model=model,
    generator=inference_generator,
    house=CONFIG['inference_house'],
    cuda=cuda_available,
    binary_threshold=CONFIG.get('binary_threshold')
)

# 获取原始数据（反缩放后）
ground_truth = inference_generator.get_target()
source = inference_generator.get_source()

print(f"\n推理结果指标:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

In [ ]:
# 可视化预测结果
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# 选取一段数据进行可视化（避免数据量过大）
start_idx = 0
end_idx = min(5000, len(predictions))  # 显示前5000个点
display_range = range(start_idx, end_idx)

# 图1: 总功率与预测对比
ax1 = axes[0]
ax1.plot(display_range, source[start_idx:end_idx], 'b-', 
         label='Aggregate Power (总功率)', alpha=0.7, linewidth=0.8)
ax1.set_xlabel('Sample Index', fontsize=11)
ax1.set_ylabel('Power (W)', fontsize=11)
ax1.set_title('Aggregate Power Consumption', fontsize=13)
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)

# 图2: 真实值与预测值对比
ax2 = axes[1]
ax2.plot(display_range, ground_truth[start_idx:end_idx], 'g-', 
         label='Ground Truth (真实值)', alpha=0.8, linewidth=1)
ax2.plot(display_range, predictions[start_idx:end_idx], 'r-', 
         label='Prediction (预测值)', alpha=0.8, linewidth=1)
ax2.set_xlabel('Sample Index', fontsize=11)
ax2.set_ylabel('Power (W)', fontsize=11)
ax2.set_title(f'Target Device: {CONFIG["target_device"]} - Ground Truth vs Prediction', fontsize=13)
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(True, alpha=0.3)

# 图3: 预测误差
ax3 = axes[2]
error = predictions[start_idx:end_idx] - ground_truth[start_idx:end_idx]
ax3.fill_between(display_range, 0, error, 
                  where=error >= 0, color='red', alpha=0.5, 
                  label='Overestimation (高估)')
ax3.fill_between(display_range, 0, error, 
                  where=error < 0, color='blue', alpha=0.5, 
                  label='Underestimation (低估)')
ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax3.set_xlabel('Sample Index', fontsize=11)
ax3.set_ylabel('Prediction Error (W)', fontsize=11)
ax3.set_title('Prediction Error Analysis', fontsize=13)
ax3.legend(loc='upper right', fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()

# 保存图片
visualization_path = os.path.join(LOG_DIR, 'prediction_visualization.png')
plt.savefig(visualization_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n预测可视化已保存到: {visualization_path}")

In [ ]:
# 详细误差统计
print("=" * 60)
print("预测效果详细统计")
print("=" * 60)

# 有效数据掩码
valid_mask = np.ones_like(source)
for i in range(len(source)):
    if source[i] == 0 or source[i] < ground_truth[i]:
        valid_mask[i] = 0

valid_predictions = predictions * valid_mask
valid_ground_truth = ground_truth * valid_mask

# 计算各项指标
mae = np.mean(np.abs(valid_predictions - valid_ground_truth))
rmse = np.sqrt(np.mean((valid_predictions - valid_ground_truth) ** 2))
mape = np.mean(np.abs((valid_ground_truth - valid_predictions) / (valid_ground_truth + 1e-8))) * 100
sae = np.abs(np.sum(valid_predictions) - np.sum(valid_ground_truth)) / np.sum(valid_ground_truth)

print(f"平均绝对误差 (MAE): {mae:.4f} W")
print(f"均方根误差 (RMSE): {rmse:.4f} W")
print(f"平均绝对百分比误差 (MAPE): {mape:.2f}%")
print(f"信号聚合误差 (SAE): {sae:.4f} ({sae*100:.2f}%)")

# 误差分布直方图
fig, ax = plt.subplots(figsize=(10, 5))
valid_errors = (valid_predictions - valid_ground_truth)[valid_mask == 1]
ax.hist(valid_errors, bins=50, color='steelblue', edgecolor='white', alpha=0.7)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
ax.axvline(x=np.mean(valid_errors), color='orange', linestyle='--', 
           linewidth=2, label=f'Mean Error: {np.mean(valid_errors):.2f}')
ax.set_xlabel('Prediction Error (W)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Prediction Errors', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
error_dist_path = os.path.join(LOG_DIR, 'error_distribution.png')
plt.savefig(error_dist_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n误差分布图已保存到: {error_dist_path}")

## 12. 保存实验结果汇总

In [ ]:
# 实验结果汇总
experiment_results = {
    'experiment_id': EXPERIMENT_ID,
    'model': CONFIG['model'],
    'model_params': CONFIG['model_params'],
    'target_device': CONFIG['target_device'],
    'best_iteration': int(best_iter),
    'best_val_mae': float(best_mae),
    'inference_metrics': metrics,
    'detailed_metrics': {
        'MAE': float(mae),
        'RMSE': float(rmse),
        'MAPE': float(mape),
        'SAE': float(sae)
    },
    'config': CONFIG,
    'best_model_path': best_model_path
}

# 保存结果
results_path = os.path.join(LOG_DIR, 'experiment_results.json')
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(experiment_results, f, indent=2, ensure_ascii=False)

print("=" * 60)
print("实验结果汇总")
print("=" * 60)
print(json.dumps(experiment_results, indent=2, ensure_ascii=False))
print(f"\n实验结果已保存到: {results_path}")

In [ ]:
# 打印所有输出文件路径
print("=" * 60)
print("输出文件汇总")
print("=" * 60)
print(f"实验ID: {EXPERIMENT_ID}")
print(f"\n模型保存目录: {MODEL_SAVE_DIR}")
print(f"  - 最优模型: {best_model_path}")
print(f"\n日志目录: {LOG_DIR}")
print(f"  - 训练曲线: {os.path.join(LOG_DIR, 'training_curves.png')}")
print(f"  - 预测可视化: {os.path.join(LOG_DIR, 'prediction_visualization.png')}")
print(f"  - 误差分布: {os.path.join(LOG_DIR, 'error_distribution.png')}")
print(f"  - 训练历史: {os.path.join(LOG_DIR, 'training_history.json')}")
print(f"  - 实验结果: {os.path.join(LOG_DIR, 'experiment_results.json')}")